# Preprocessing & Feature Engineering (v1 base model prototype)

Prototype of plan step 3 in pandas, on a small local sample. The finished logic is meant to be ported 1:1 to the PySpark Processing job, so all logic lives in small pure functions.

**Steps:** load parquet → null/duplicate handling → past-only account counts → filter to `TRANSFER`/`CASH_OUT` → time/type features → leaky-column gate → time-based split by `step` → class-imbalance weight → persist locally.

References: `DOCS/feature-engineering.md`, `DOCS/dataset-info.md`. All thresholds come from `config/preprocessing/preprocessing.yaml`.

# 1. Setup

## 1.1 Imports, logging and repo root

Imports, a module-level logger, and a helper that locates the repo root so paths in the config resolve no matter where Jupyter was started.

In [ ]:
import json
import logging
import subprocess
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import yaml

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
logger = logging.getLogger("preprocessing_nb")


def find_repo_root(start: Path | None = None) -> Path:
    """Walks up from ``start`` to the directory containing ``pyproject.toml``.

    Args:
        start: Directory to start from. Defaults to the current working directory.

    Returns:
        Path of the repository root.

    Raises:
        FileNotFoundError: If no ancestor contains ``pyproject.toml``.
    """
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(f"pyproject.toml not found above {start}")


REPO_ROOT = find_repo_root()
logger.info("step=setup repo_root=%s", REPO_ROOT)

2026-09-24 22:32:33,183 INFO preprocessing_nb step=setup repo_root=F:\git-projects\transaction-fraud-detection


## 1.2 Load configuration

All tunables (input path, type filter, night hours, split fractions, the leaky-feature flag) are read from YAML instead of being hardcoded, per the repo's config-driven rule.

In [ ]:
def load_config(path: Path) -> dict[str, Any]:
    """Loads a YAML config file.

    Args:
        path: Path to the YAML file.

    Returns:
        Parsed configuration as a nested dict.
    """
    with path.open() as f:
        return yaml.safe_load(f)


CONFIG_PATH = REPO_ROOT / "config" / "preprocessing" / "preprocessing.yaml"
config = load_config(CONFIG_PATH)
LABEL = config["data"]["label_col"]
logger.info("step=config status=loaded path=%s", CONFIG_PATH)
config

2026-09-24 22:32:33,191 INFO preprocessing_nb step=config status=loaded path=F:\git-projects\transaction-fraud-detection\configs\preprocessing\preprocessing.yaml


{'data': {'input_path': 'dataset/raw/data-v0',
  'output_dir': 'dataset/processed/data-v0',
  'label_col': 'isFraud',
  'keep_types': ['TRANSFER', 'CASH_OUT']},
 'features': {'include_balance_error_features': False, 'night_hours': [0, 6]},
 'split': {'train_frac_of_max_step': 0.7, 'val_frac_of_max_step': 0.85},
 'seed': 42}

# 2. Load raw data

Read the Hive-partitioned parquet with the PyArrow Dataset API (it parses `day=` folders natively and supports column/filter pushdown for the full 15GB+ run), select only the raw columns, and validate the schema, so upstream changes fail loudly here rather than deep in feature code.

In [ ]:
RAW_COLUMNS = [
    "step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud",
]


def load_raw(path: Path) -> pd.DataFrame:
    """Reads the raw parquet dataset and validates its columns.

    Args:
        path: Directory (or file) holding the Hive-partitioned parquet data.

    Returns:
        DataFrame with exactly the raw columns (partition column dropped).

    Raises:
        ValueError: If any expected raw column is missing.
    """
    dataset = ds.dataset(path, format="parquet", partitioning="hive")
    missing = set(RAW_COLUMNS) - set(dataset.schema.names)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    return dataset.to_table(columns=RAW_COLUMNS).to_pandas()


raw = load_raw(REPO_ROOT / config["data"]["input_path"])
logger.info(
    "step=load status=complete rows=%d fraud_rate=%.4f step_min=%d step_max=%d",
    len(raw), raw[LABEL].mean(), raw["step"].min(), raw["step"].max(),
)
raw.dtypes

2026-09-24 22:32:33,288 INFO preprocessing_nb step=load status=complete rows=8 fraud_rate=0.5000 step_min=1 step_max=12


step                       int32
type              string[python]
amount                   float64
nameOrig          string[python]
oldbalanceOrg            float64
newbalanceOrig           float64
nameDest          string[python]
oldbalanceDest           float64
newbalanceDest           float64
isFraud                     int8
isFlaggedFraud              int8
dtype: object

# 3. Data quality

## 3.1 Nulls, duplicates and zero amounts

The source has no NaNs, but we still drop rows with nulls in required columns (cheap safeguard) and remove exact duplicate rows. Zero-amount rows are only logged for inspection. Merchant `0.0` balances are *not* imputed: they mean "no information", not zero.

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Drops null-containing and exact-duplicate rows and reports quality stats.

    Args:
        df: Raw transactions.

    Returns:
        Cleaned DataFrame with a reset index, original row order preserved.
    """
    null_counts = df.isna().sum()
    logger.info("step=clean nulls_total=%d", int(null_counts.sum()))
    out = df.dropna()
    n_dupes = int(out.duplicated().sum())
    out = out.drop_duplicates()
    logger.info(
        "step=clean rows_in=%d rows_out=%d duplicates_dropped=%d zero_amount_rows=%d",
        len(df), len(out), n_dupes, int((out["amount"] == 0).sum()),
    )
    return out.reset_index(drop=True)


clean_df = clean(raw)

2026-09-24 22:32:33,302 INFO preprocessing_nb step=clean nulls_total=0


2026-09-24 22:32:33,312 INFO preprocessing_nb step=clean rows_in=8 rows_out=8 duplicates_dropped=0 zero_amount_rows=0


# 4. Feature engineering

## 4.1 Past-only account counts (before the type filter)

`orig_txn_count` / `dest_txn_count` count how many transactions the account has made / received **up to and including the current one**, using only rows with earlier-or-equal `step`. This avoids future leakage and counts over *all* transaction types, so it must run **before** the type filter.

Ordering is `step` then original row position (stable), so ties are deterministic. PySpark equivalent: `count(*)` over `Window.partitionBy(name).orderBy(step, row_id).rowsBetween(unboundedPreceding, currentRow)`.

In [ ]:
def add_past_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Adds past-only, inclusive running transaction counts per account.

    Args:
        df: Transactions with ``step``, ``nameOrig`` and ``nameDest``.

    Returns:
        Copy of ``df`` (sorted by ``step``, stable) with ``orig_txn_count`` and
        ``dest_txn_count`` columns.
    """
    out = df.sort_values("step", kind="stable").reset_index(drop=True)
    out["orig_txn_count"] = out.groupby("nameOrig").cumcount() + 1
    out["dest_txn_count"] = out.groupby("nameDest").cumcount() + 1
    return out


counted = add_past_counts(clean_df)
counted[["step", "nameOrig", "nameDest", "orig_txn_count", "dest_txn_count"]]

,step,nameOrig,nameDest,orig_txn_count,dest_txn_count
0,1,C1305486145,C553264065,1,1
1,1,C840083671,C38997010,1,1
2,4,C1912850431,C1590550415,1,1
3,5,C1670993182,C1100697970,1,1
4,8,C1976401987,C1937962514,1,1
5,9,C905080434,C476402209,1,1
6,12,C1984094095,C932583850,1,1
7,12,C1984094095,C1023714065,2,1


## 4.2 Filter to `TRANSFER` and `CASH_OUT`

Fraud only occurs in these two types, so the base model trains on them only. Serving must route other types to a rule or reject them.

In [ ]:
def filter_types(df: pd.DataFrame, keep_types: list[str]) -> pd.DataFrame:
    """Keeps only the configured transaction types.

    Args:
        df: Transactions with a ``type`` column.
        keep_types: Transaction types to retain.

    Returns:
        Filtered DataFrame with a reset index.
    """
    out = df[df["type"].isin(keep_types)].reset_index(drop=True)
    logger.info(
        "step=filter rows_in=%d rows_out=%d fraud_in=%d fraud_out=%d",
        len(df), len(out), int(df[LABEL].sum()), int(out[LABEL].sum()),
    )
    return out


filtered = filter_types(counted, config["data"]["keep_types"])

2026-09-24 22:32:33,340 INFO preprocessing_nb step=filter rows_in=8 rows_out=8 fraud_in=4 fraud_out=4


## 4.3 Type, amount and time features

Adds `is_transfer` (the only categorical encoding needed: `type` has two values left), `log_amount`, `hour_of_day`, `is_night` and `day_of_month`. The night window comes from config.

In [ ]:
def add_basic_features(df: pd.DataFrame, night_hours: list[int]) -> pd.DataFrame:
    """Adds type, amount and time-derived features.

    Args:
        df: Filtered transactions with ``type``, ``amount`` and ``step``.
        night_hours: ``[start, end]`` inclusive hour range flagged as night.

    Returns:
        Copy of ``df`` with ``is_transfer``, ``log_amount``, ``hour_of_day``,
        ``is_night`` and ``day_of_month`` added.
    """
    out = df.copy()
    out["is_transfer"] = (out["type"] == "TRANSFER").astype("int8")
    out["log_amount"] = np.log1p(out["amount"])
    out["hour_of_day"] = (out["step"] % 24).astype("int16")
    out["is_night"] = out["hour_of_day"].between(night_hours[0], night_hours[1]).astype("int8")
    out["day_of_month"] = (out["step"] // 24).astype("int16")
    return out


featured = add_basic_features(filtered, config["features"]["night_hours"])
featured[["step", "type", "amount", "is_transfer", "log_amount", "hour_of_day", "is_night", "day_of_month"]]

,step,type,amount,is_transfer,log_amount,hour_of_day,is_night,day_of_month
0,1,TRANSFER,181.00,1,5.204007,1,1,0
1,1,CASH_OUT,181.00,0,5.204007,1,1,0
2,4,CASH_OUT,136872.00,0,11.826809,4,1,0
3,5,TRANSFER,215310.30,1,12.279840,5,1,0
4,8,TRANSFER,62610.80,1,11.044709,8,0,0
5,9,CASH_OUT,229133.94,0,12.342066,9,0,0
6,12,TRANSFER,311685.89,1,12.649754,12,0,0
7,12,CASH_OUT,311685.89,0,12.649754,12,0,0


## 4.4 Leaky-column gate

`newbalanceOrig` / `newbalanceDest` are post-transaction state and are **always dropped**. The `errorBalance*` features are built from them, so they only exist when `features.include_balance_error_features` is true (default false; a PaySim artifact that will not transfer to real data). The raw ID strings, `type`, `isFlaggedFraud` and all raw balance columns are dropped from the model output in every case.

In [ ]:
BASE_FEATURES = [
    "is_transfer", "amount", "log_amount", "hour_of_day",
    "is_night", "day_of_month", "orig_txn_count", "dest_txn_count",
]
ERROR_FEATURES = ["errorBalanceOrig", "errorBalanceDest"]


def apply_leakage_gate(df: pd.DataFrame, include_error_features: bool) -> tuple[pd.DataFrame, list[str]]:
    """Optionally builds balance-error features, then keeps only model columns.

    Args:
        df: Featured transactions still holding the raw balance columns.
        include_error_features: If True, add ``errorBalanceOrig`` and
            ``errorBalanceDest`` (leaky; for experimentation only).

    Returns:
        Tuple of (DataFrame with ``step``, the model features and the label only,
        ordered list of model feature names).
    """
    out = df.copy()
    features = list(BASE_FEATURES)
    if include_error_features:
        out["errorBalanceOrig"] = out["oldbalanceOrg"] - out["amount"] - out["newbalanceOrig"]
        out["errorBalanceDest"] = out["newbalanceDest"] - out["oldbalanceDest"] - out["amount"]
        features += ERROR_FEATURES
        logger.warning("step=leakage_gate include_error_features=true model_will_use_leaky_features")
    out = out[["step", *features, LABEL]]
    logger.info("step=leakage_gate n_features=%d features=%s", len(features), features)
    return out, features


model_df, FEATURES = apply_leakage_gate(featured, config["features"]["include_balance_error_features"])
model_df

2026-09-24 22:32:33,368 INFO preprocessing_nb step=leakage_gate n_features=8 features=['is_transfer', 'amount', 'log_amount', 'hour_of_day', 'is_night', 'day_of_month', 'orig_txn_count', 'dest_txn_count']


,step,is_transfer,amount,log_amount,hour_of_day,is_night,day_of_month,orig_txn_count,dest_txn_count,isFraud
0,1,1,181.00,5.204007,1,1,0,1,1,1
1,1,0,181.00,5.204007,1,1,0,1,1,1
2,4,0,136872.00,11.826809,4,1,0,1,1,0
3,5,1,215310.30,12.279840,5,1,0,1,1,0
4,8,1,62610.80,11.044709,8,0,0,1,1,0
5,9,0,229133.94,12.342066,9,0,0,1,1,0
6,12,1,311685.89,12.649754,12,0,0,1,1,1
7,12,0,311685.89,12.649754,12,0,0,2,1,1


# 5. Time-based train / val / test split

A random split would let the model see the "future" half of a TRANSFER→CASH_OUT fraud pair. Instead we cut on `step` using thresholds derived from the max step (fractions in config): train `step <= t1`, val `t1 < step <= t2`, test `step > t2`. Plain boolean masks, so it ports directly to a Spark `filter`. Anything fitted (the class weight below) uses train only.

In [ ]:
def time_split(
    df: pd.DataFrame, train_frac: float, val_frac: float
) -> tuple[dict[str, pd.DataFrame], dict[str, int]]:
    """Splits rows into train/val/test by ``step`` thresholds.

    Args:
        df: Model DataFrame containing ``step`` and the label.
        train_frac: Fraction of the max step where the train split ends.
        val_frac: Fraction of the max step where the validation split ends.

    Returns:
        Tuple of (dict with ``train``/``val``/``test`` DataFrames, dict with the
        ``train_max_step`` and ``val_max_step`` thresholds).

    Raises:
        ValueError: If the fractions are not ``0 < train_frac < val_frac < 1``.
        AssertionError: If the splits drop or duplicate rows.
    """
    if not 0 < train_frac < val_frac < 1:
        raise ValueError("require 0 < train_frac < val_frac < 1")
    max_step = int(df["step"].max())
    t1, t2 = int(np.floor(train_frac * max_step)), int(np.floor(val_frac * max_step))
    splits = {
        "train": df[df["step"] <= t1],
        "val": df[(df["step"] > t1) & (df["step"] <= t2)],
        "test": df[df["step"] > t2],
    }
    assert sum(len(s) for s in splits.values()) == len(df), "split lost or duplicated rows"
    for name, part in splits.items():
        n_pos = int(part[LABEL].sum())
        logger.info("step=split part=%s rows=%d fraud=%d", name, len(part), n_pos)
        if len(part) == 0 or n_pos == 0:
            logger.warning("step=split part=%s has_no_rows_or_no_positives=true", name)
    return splits, {"train_max_step": t1, "val_max_step": t2}


splits, thresholds = time_split(model_df, config["split"]["train_frac_of_max_step"], config["split"]["val_frac_of_max_step"])
thresholds

2026-09-24 22:32:33,383 INFO preprocessing_nb step=split part=train rows=5 fraud=2


2026-09-24 22:32:33,384 INFO preprocessing_nb step=split part=val rows=1 fraud=0


2026-09-24 22:32:33,384 WARNING preprocessing_nb step=split part=val has_no_rows_or_no_positives=true


2026-09-24 22:32:33,385 INFO preprocessing_nb step=split part=test rows=2 fraud=2


{'train_max_step': 8, 'val_max_step': 10}

# 6. Class imbalance

Fraud is ~0.13% of the full data, so accuracy is meaningless. We do **not** resample here (resampling belongs inside training/CV folds). Instead we compute `scale_pos_weight = n_neg / n_pos` from the **train split only** for XGBoost. If train has no positives the weight is `None`. On the tiny `data-v0` sample the ~50% fraud rate makes this value uninformative.

In [ ]:
def compute_scale_pos_weight(train: pd.DataFrame) -> float | None:
    """Computes XGBoost's ``scale_pos_weight`` from the training split.

    Args:
        train: Training split containing the label column.

    Returns:
        ``n_negative / n_positive``, or None when there are no positives.
    """
    n_pos = int(train[LABEL].sum())
    n_neg = len(train) - n_pos
    if n_pos == 0:
        logger.warning("step=imbalance train_has_no_positives=true")
        return None
    weight = n_neg / n_pos
    logger.info("step=imbalance n_pos=%d n_neg=%d scale_pos_weight=%.4f", n_pos, n_neg, weight)
    return weight


scale_pos_weight = compute_scale_pos_weight(splits["train"])
scale_pos_weight

2026-09-24 22:32:33,392 INFO preprocessing_nb step=imbalance n_pos=2 n_neg=3 scale_pos_weight=1.5000


1.5

# 7. Persist locally

Write each split as a parquet folder (PyArrow, Spark-compatible) under the configured output dir, plus `metadata.json` capturing thresholds, feature list, class weight, row counts, source path, the git commit and the config snapshot, so the produced dataset can be traced back to what created it.

In [ ]:
def git_commit() -> str:
    """Returns the current git commit hash, or ``unknown`` if unavailable.

    Returns:
        Short commit hash string.
    """
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True
        ).strip()
    except (subprocess.SubprocessError, OSError):
        return "unknown"


def persist(splits: dict[str, pd.DataFrame], out_dir: Path, metadata: dict[str, Any]) -> None:
    """Writes each split as a parquet folder (via PyArrow) and the metadata as JSON.

    Files are written as ``part-{i}.parquet`` inside one folder per split, the
    same layout Spark produces, so readers must load the folder, not a filename.

    Args:
        splits: Mapping of split name to DataFrame.
        out_dir: Destination directory (created if needed).
        metadata: JSON-serialisable lineage information.
    """
    for name, part in splits.items():
        target = out_dir / name
        target.mkdir(parents=True, exist_ok=True)
        ds.write_dataset(
            pa.Table.from_pandas(part, preserve_index=False),
            target,
            format="parquet",
            basename_template="part-{i}.parquet",
            existing_data_behavior="overwrite_or_ignore",
        )
        logger.info("step=persist part=%s rows=%d path=%s", name, len(part), target)
    (out_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))


OUT_DIR = REPO_ROOT / config["data"]["output_dir"]
metadata = {
    "source_path": config["data"]["input_path"],
    "git_commit": git_commit(),
    "features": FEATURES,
    "label": LABEL,
    "thresholds": thresholds,
    "scale_pos_weight": scale_pos_weight,
    "row_counts": {k: len(v) for k, v in splits.items()},
    "fraud_counts": {k: int(v[LABEL].sum()) for k, v in splits.items()},
    "config": config,
}
persist(splits, OUT_DIR, metadata)

2026-09-24 22:32:33,446 INFO preprocessing_nb step=persist part=train rows=5 path=F:\git-projects\transaction-fraud-detection\dataset\processed\data-v0\train


2026-09-24 22:32:33,459 INFO preprocessing_nb step=persist part=val rows=1 path=F:\git-projects\transaction-fraud-detection\dataset\processed\data-v0\val


2026-09-24 22:32:33,463 INFO preprocessing_nb step=persist part=test rows=2 path=F:\git-projects\transaction-fraud-detection\dataset\processed\data-v0\test


# 8. Sanity checks

Reload the persisted files and assert the contract the training step will rely on: expected columns, no NaNs, no leaky columns, ordered non-overlapping step ranges.

In [ ]:
LEAKY = {
    "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
    "nameOrig", "nameDest", "type", "isFlaggedFraud",
}


def validate_outputs(out_dir: Path, features: list[str]) -> None:
    """Reloads persisted splits and asserts the output contract.

    Args:
        out_dir: Directory the splits were written to.
        features: Expected model feature names.

    Raises:
        AssertionError: If columns, nulls, leakage or step ordering are wrong.
    """
    loaded = {n: ds.dataset(out_dir / n, format="parquet").to_table().to_pandas() for n in ("train", "val", "test")}
    for name, part in loaded.items():
        assert list(part.columns) == ["step", *features, LABEL], f"{name}: unexpected columns"
        assert not part.isna().any().any(), f"{name}: contains NaN"
        assert not LEAKY & set(part.columns), f"{name}: leaky column present"
    assert loaded["val"].empty or loaded["train"]["step"].max() < loaded["val"]["step"].min()
    assert loaded["test"].empty or loaded["val"]["step"].max() < loaded["test"]["step"].min()
    logger.info("step=validate status=ok n_features=%d", len(features))


validate_outputs(OUT_DIR, FEATURES)